In [ ]:
import pandas as pd

In [ ]:
df = pd.read_json("op.json",encoding="utf-16")

In [ ]:
df.head(15)

In [ ]:
counts = df.groupby('author').size()
counts[counts>2]
result = df[df['author'].isin(counts[counts > 2].index)]

In [ ]:
result[['title','selftext']]

In [ ]:
result.query("selftext != '[removed]'").index

In [ ]:
intermediary_df = result.query("selftext != '[removed]' and selftext.str.len() > 2")[['author','title','selftext']]

In [ ]:
intermediary_df

In [ ]:
import re
def clean_and_spellcheck(text):
    if not isinstance(text, str):
        return []
    
    # Separate glued letters and numbers (e.g., '2929\n\n\nThat's' or 'mental14')
    text = re.sub(r'([a-zA-Z]+)(\d+)', r'\1 \2', text)
    
    # Normalize curly apostrophes to straight apostrophes
    text = text.replace("’", "'")
    
    # Extract clean alphabetic tokens
    clean_tokens = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text.lower())
    
    return " ".join(clean_tokens)
intermediary_df['cleaned_selftext'] = intermediary_df['selftext'].progress_apply(clean_and_spellcheck)

In [ ]:
import joblib
import torch
import torch.nn as nn
from tqdm import tqdm

# Load vectorizer
vectorizer = joblib.load("tfidf_vectorizer.pkl")

input_size = len(vectorizer.get_feature_names_out())

tqdm.pandas()

model = nn.Sequential(
    nn.Linear(input_size, 16),
    nn.ReLU(),
    nn.Linear(16, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)

# Load trained weights
model.load_state_dict(
    torch.load(
        "suicide_classifier.pth",
        map_location="cpu",
        weights_only=True
    )
)

# Load model
model.load_state_dict(torch.load("suicide_classifier.pth", weights_only=True))
model.eval()

# new_text = ["""
# i just wanna give up. I'm tired.

# i don't have money but I guess I could find a way to get a bottle of silver cleaner. i just want everything to end.
# """]

def pred(new_text:str):
    X_new = vectorizer.transform([new_text])

    X_new_tensor = torch.tensor(X_new.toarray(), dtype=torch.float32)

    with torch.no_grad():
        logits = model(X_new_tensor)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

    # print("Probability:", probabilities.item())
    return("Prediction:", predictions.item())


intermediary_df['suicidal_text'] = intermediary_df['cleaned_selftext'].progress_apply(pred)

In [ ]:
intermediary_df.groupby('author').size()

In [ ]:
list(df.columns)

In [ ]:
list(user_df.columns)

In [ ]:
user_df = pd.read_json('u_yaoimalover101_posts.jsonl',lines=True)

In [ ]:
user_df[['subreddit','title','selftext','created_utc']]

In [ ]:
user_df[['subreddit','title','selftext','created_utc']].query("subreddit == 'SuicideWatch'")

In [ ]:
filtered_df = user_df[['subreddit', 'title', 'selftext', 'created_utc']].copy()
filtered_df['created_ist'] = (
    pd.to_datetime(filtered_df['created_utc'], unit='s', utc=True)
    .dt.tz_convert('Asia/Kolkata')
)

In [ ]:
filtered_df